# 10-8 雜湊容器綜合實戰：座標標記、稀疏網格與圖論前導 (Hash Containers Practice)

恭喜大家一路跋涉，來到了第十章的終章！在前面的章節中，我們依序精通了：
- **元組（Tuple）**：輕量不可變、支援多欄位打包與座標封裝 `(r, c)`。
- **字典（Dictionary）**：以任意可雜湊鍵建立名牌映射，提供 $O(1)$ 極速查表與分組。
- **集合（Set）**：天然去重、支援強大的范氏圖代數運算（`&`, `|`, `-`, `^`）。

在 APCS 的二星與三星進階題目中，這三大雜湊容器往往不是單獨作戰，而是**深度聯手、協同出擊**！例如：用「元組」封裝座標，丟進「集合」中標記足跡；用「字典套串列」建構「圖論鄰接串列」；用「元組鍵值對調」在不使用自訂函式的情況下實現多條件排序。

本節將透過**極致緩坡架構**，把前七節的知識精髓融會貫通，帶你跨越基礎語法，真正具備迎戰 APCS 實作真題的硬核演算法戰力！

### 🎯 本節學習目標
1. 掌握「元組 + 集合」：以 `(r, c)` 集合標記已拜訪足跡，省下 99% 陣列記憶體。
2. 掌握「元組 + 字典」：以 `{(r, c): val}` 實現動態稀疏網格，終結超大地圖 MLE 困境。
3. 掌握「字典 + 串列」：以字典套串列實作「圖論鄰接串列（Adjacency List）」。
4. 掌握「元組鍵值對調排序法」：在零自訂函式（零 def、零 lambda）下實現多準則排序。
5. 掌握「單趟雜湊掃描（One-pass Hash Scan）」：在 $O(N)$ 時間內鎖定首個重複值。
6. 串聯 APCS 歷屆真題原型（g276 魔王迷宮、c291 小群體），體會雜湊思維的實戰威力。

> ⚠️ **溫馨提醒**：本節程式碼嚴格恪守「零提前依賴」，全篇完全不使用自訂函式 `def` 或匿名函式 `lambda`，純用 Python 原生資料結構展現演算法之美！

## 10.8.1 元組作為集合元素：以 (r, c) 集合實現稀疏地圖已拜訪標記（Visited Set）

### 觀念說明
在第 9 章中，我們學習了二維網格。若要記錄哪些格子已經走過，傳統寫法是開一個布林矩陣：
`visited = [[False] * C for _ in range(R)]`。

但是，如果題目的網格邊界高達 $R = 100,000, C = 100,000$，宣告這個二維矩陣需要一百億個布林值，記憶體會瞬間爆表（引發 MLE）！然而，題目往往規定機器人最多只走 1,000 步。

### 集合足跡標記法（Visited Set）
這時最佳的解法，就是利用**「元組作為集合的元素」**：
1. 宣告空集合：`visited = set()`。
2. 每當走到一個新座標時，以元組打包並加入集合：`visited.add((r, c))`。
3. 檢查下一步是否走過：`if (nr, nc) in visited:`，雜湊查詢平均只需 **$O(1)$**！

記憶體消耗只與「實際走過的步數」成正比，省下了數億倍的記憶體空間，是解決大地圖模擬問題的必備神技！

In [ ]:
# 10.8.1 範例：機器人百萬地圖漫步與回頭路偵測
# 機器人從 (0, 0) 開始出發，移動指令為 (dr, dc)
moves = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 1)]

curr_r, curr_c = 0, 0
visited = set()
visited.add((curr_r, curr_c))

print(f"起點: ({curr_r}, {curr_c})")

for dr, dc in moves:
    next_r = curr_r + dr
    next_c = curr_c + dc
    next_pt = (next_r, next_c)
    
    if next_pt in visited:
        print(f"警告！座標 {next_pt} 曾經走過，重複踩踏！")
    else:
        visited.add(next_pt)
        print(f"抵達新座標: {next_pt}")
        
    curr_r, curr_c = next_r, next_c

print("總共踩過的不同座標數:", len(visited))

In [ ]:
# 10.8.1 填空題：將起點座標元組加入已拜訪集合
visited_points = set()
start_r, start_c = 3, 5

# 任務：以元組 (start_r, start_c) 的形式加入 visited_points 集合中
visited_points.add((start_r, start_c))

# 請填入空格：
# visited_points.add((___, ___))
print("已拜訪集合內容:", visited_points)
# 預期輸出: 已拜訪集合內容: {(3, 5)}

In [ ]:
# 10.8.1 練習題：探險家足跡覆蓋統計
# 題目說明：
# 探險家從 (0, 0) 起點出發，依序執行移動路徑 path（每步為向量 (dr, dc)）。
# 請記錄探險家走過的所有「相異座標」，並輸出：
# 1. 探險家總共走訪了多少個不同的座標點？
# 2. 探險家最終停留在哪一個座標點 (final_r, final_c)？
#
# 【公開測試資料 1】
# 移動指令: [(1, 0), (1, 0), (0, 1), (-1, 0)]
# 走過座標: (0,0) -> (1,0) -> (2,0) -> (2,1) -> (1,1)
# 輸出:
# 相異座標數: 5
# 最終終點: (1, 1)
#
# 【公開測試資料 2】
# 移動指令: [(0, 1), (0, -1), (0, 1), (0, -1)]  # 來回走動
# 走過座標: (0,0), (0,1)
# 輸出:
# 相異座標數: 2
# 最終終點: (0, 0)

path = [(1, 0), (1, 0), (0, 1), (-1, 0)]

r, c = 0, 0
visited = {(r, c)}

for dr, dc in path:
    r += dr
    c += dc
    visited.add((r, c))

print("相異座標數:", len(visited))
print("最終終點:", (r, c))

In [ ]:
# 10.8.1 挑戰題：首度自交點（First Self-Intersection）偵測
# 題目說明：
# 貪食蛇從 (0, 0) 開始移動，依序前進 moves = [(1,0), (0,1), (-1,0), (0,-1), (1,0)]
# 當蛇移動到一個「曾經踏足過」的座標時，稱為發生自交（碰觸自己身體）。
# 請輸出第一次發生自交的座標點與前進步數（第幾步發生）；若從未發生自交，輸出 "平安無事"。

moves = [(1, 0), (0, 1), (-1, 0), (0, -1), (1, 0)]

# 請在下方寫出你的解答代碼
r, c = 0, 0
visited = {(r, c)}
intersect_pt = None
step_count = 0

for dr, dc in moves:
    step_count += 1
    r += dr
    c += dc
    if (r, c) in visited:
        intersect_pt = (r, c)
        break
    visited.add((r, c))

if intersect_pt:
    print(f"於第 {step_count} 步發生自交，座標為: {intersect_pt}")
else:
    print("平安無事")
# 預期結果: 於第 4 步發生自交，座標為: (0, 0)

## 10.8.2 元組作為字典鍵值：以 {(r, c): value} 實現動態稀疏網格（省去百萬記憶體）

### 觀念說明
在上一個子單元中，集合只能記錄座標「有沒有走過（布林狀態）」。
但如果地圖上的每一個特定點位，都有「不同的道具、數值或怪獸資訊」，該怎麼辦？

### 稀疏網格（Sparse Grid）模型
在大多數遊戲地圖或競技題中，地圖可能非常大（例如 $10^5 \times 10^5$），但其中真正有放寶物的格子只有少少幾十個。絕大多數格子都是空的（數值為 0）。

此時，「**以元組 `(r, c)` 作為字典的 Key**」就是最高效的稀疏網格表示法：
```python
sparse_grid = {
    (100, 200): "寶箱",
    (5000, 8888): "巨龍"
}
```
### 安全讀取心法
當要查詢某個座標 `(r, c)` 時，搭配 `grid.get((r, c), default_val)`（例如 `grid.get((r, c), 0)`）。如果該座標沒有寶物，直接回傳預設值 0，完全不需要預先初始化龐大的二維陣列！

In [ ]:
# 10.8.2 範例：大地圖稀疏金幣收集器
# 建立稀疏網格，只記錄有金幣的座標點
gold_map = {
    (0, 5): 100,
    (10, 20): 500,
    (999, 999): 1000
}

# 冒險者拜訪一系列座標
tour_coords = [(0, 5), (1, 1), (10, 20), (50, 50)]

collected_gold = 0
for pt in tour_coords:
    # 使用 get()，若沒金幣預設為 0
    gold = gold_map.get(pt, 0)
    print(f"走訪座標 {pt} ➔ 獲得金幣: {gold}")
    collected_gold += gold

print("總計收穫金幣:", collected_gold)

In [ ]:
# 10.8.2 填空題：在稀疏地圖上放置與讀取旗幟
world_map = {}

flag_pos = (12, 34)
# 任務：在 world_map 上的 flag_pos 放置 "FLAG_A"
world_map[flag_pos] = "FLAG_A"

# 任務：使用 get 查詢 (12, 34) 與 (0, 0)，若無旗幟預設為 "EMPTY"
res1 = world_map.get((12, 34), "EMPTY")
res2 = world_map.get((0, 0), "EMPTY")

# 請填入空格：
# res1 = world_map.get(___, "EMPTY")
# res2 = world_map.get(___, "EMPTY")
print("(12, 34) 狀態:", res1)
print("(0, 0) 狀態:", res2)
# 預期輸出:
# (12, 34) 狀態: FLAG_A
# (0, 0) 狀態: EMPTY

In [ ]:
# 10.8.2 練習題：稀疏棋盤子力總價值計算器
# 題目說明：
# 棋盤上散佈著若干棋子，每顆棋子的位置為 (r, c)，價值為 val。
# 給定棋子登錄資料 pieces_data = [((1, 2), 5), ((3, 4), 9), ((1, 2), 3)]
# 請注意：若同一座標重複放置棋子，後者的價值會「累加」上去！
# 請使用字典建立稀疏棋盤，並在最後計算整個棋盤上棋子的總價值。
#
# 【公開測試資料 1】
# 棋子資料: [((1, 2), 5), ((3, 4), 9), ((1, 2), 3)]
# 座標 (1, 2) 累加為 8，座標 (3, 4) 為 9
# 輸出:
# 棋子總格數: 2
# 全盤總價值: 17
#
# 【公開測試資料 2】
# 棋子資料: [((0, 0), 10), ((100, 100), 20)]
# 輸出:
# 棋子總格數: 2
# 全盤總價值: 30

pieces_data = [((1, 2), 5), ((3, 4), 9), ((1, 2), 3)]

board = {}
for pos, val in pieces_data:
    board[pos] = board.get(pos, 0) + val

print("棋子總格數:", len(board))
print("全盤總價值:", sum(board.values()))

In [ ]:
# 10.8.2 挑戰題：稀疏踩地雷相鄰地雷計數
# 題目說明：
# 大地圖上有三顆地雷，座標分別為 (1, 1), (1, 2), (2, 2)
# 給定查詢座標 target = (1, 1)
# 請利用方向向量 dr, dc（四方向：上下左右），走訪 target 的四個相鄰座標，
# 統計 target 的相鄰四格中一共有幾顆地雷？（地雷集合以 set 儲存）

mine_locations = {(1, 1), (1, 2), (2, 2)}
target = (1, 1)

# 請在下方寫出你的解答代碼
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
adjacent_mine_count = 0

tr, tc = target
for dr, dc in directions:
    nr, nc = tr + dr, tc + dc
    if (nr, nc) in mine_locations:
        adjacent_mine_count += 1

print("相鄰四格地雷數:", adjacent_mine_count)
# 預期結果: 相鄰四格地雷數: 1 （只有 (1, 2) 在其右側相鄰）

## 10.8.3 字典模擬圖論鄰接串列（Adjacency List）：graph[u] = [v1, v2]

### 觀念說明
在 APCS 考題中，「圖論（Graph）」是劃分程式高手與初學者的重大分水嶺（例如社群網路好友圈、地鐵航線轉乘、APCS c291 小群體）。
在電腦中，一個「圖」由節點（Vertices）與邊（Edges）組成。

### 鄰接串列（Adjacency List）神級結構：字典套串列
表示圖最靈活、最 Pythonic 的結構就是「**字典套串列**」：
```python
graph = {}
```
- **Key**：代表當前節點 $u$。
- **Value**：是一個串列，存放所有與 $u$ 相連的鄰居節點 `[v1, v2, ...]`。

### 加邊標準模式
當讀入一條雙向邊 $(u, v)$ 時：
```python
if u not in graph: graph[u] = []
graph[u].append(v)

if v not in graph: graph[v] = []
graph[v].append(u)
```
節點名稱可以是任意字串（如 `"Taipei"`）或整數，完全不需要固定 $0$ 到 $N-1$ 的編號，極度自由強大！

In [ ]:
# 10.8.3 範例：社群好友網路圖建置
friendships = [
    ("Alice", "Bob"),
    ("Bob", "Charlie"),
    ("Alice", "David"),
    ("David", "Eva")
]

social_graph = {}
for p1, p2 in friendships:
    # 建立雙向好友邊
    if p1 not in social_graph:
        social_graph[p1] = []
    social_graph[p1].append(p2)
    
    if p2 not in social_graph:
        social_graph[p2] = []
    social_graph[p2].append(p1)

print("--- 完整鄰接串列網路 ---")
for person in sorted(social_graph.keys()):
    print(f"{person} 的好友名單: {social_graph[person]}")

print()
# 查詢特定人的好友數量（度數 Degree）
print("Alice 的好友數量:", len(social_graph["Alice"]))

In [ ]:
# 10.8.3 填空題：單向有向圖建置
flights = [("TPE", "NRT"), ("TPE", "BKK"), ("NRT", "LAX")]

flight_graph = {}
for src, dst in flights:
    if src not in flight_graph:
        flight_graph[src] = []
    flight_graph[src].append(dst)

# 請填入空格：
# flight_graph[src].append(___)
print("台北 (TPE) 可直飛航點:", flight_graph.get("TPE", []))
# 預期輸出: 台北 (TPE) 可直飛航點: ['NRT', 'BKK']

In [ ]:
# 10.8.3 練習題：公車轉乘路線鄰接表分析
# 題目說明：
# 給定站點連接紀錄 connections = [("Station_A", "Station_B"), ("Station_B", "Station_C"), ("Station_A", "Station_D")]
# 請建立雙向連通的公車網絡圖 bus_network。
# 隨後給定查詢站點 query_station = "Station_B"。
# 請輸出該站點可直接轉乘的所有相鄰站點清單（依字母排序）。
#
# 【公開測試資料 1】
# 連接: [("Station_A", "Station_B"), ("Station_B", "Station_C"), ("Station_A", "Station_D")]
# 查詢: "Station_B"
# 輸出: 相鄰站點: ['Station_A', 'Station_C']
#
# 【公開測試資料 2】
# 連接: [("X", "Y"), ("Y", "Z")]
# 查詢: "X"
# 輸出: 相鄰站點: ['Y']

connections = [("Station_A", "Station_B"), ("Station_B", "Station_C"), ("Station_A", "Station_D")]
query_station = "Station_B"

bus_network = {}
for u, v in connections:
    if u not in bus_network:
        bus_network[u] = []
    bus_network[u].append(v)
    
    if v not in bus_network:
        bus_network[v] = []
    bus_network[v].append(u)

adjacent = sorted(bus_network.get(query_station, []))
print("相鄰站點:", adjacent)

In [ ]:
# 10.8.3 挑戰題：尋找二度好友（朋友的朋友，排除自己與既有好友）
# 題目說明：
# 給定社交網路 graph = {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A', 'D', 'E'], 'D': ['B', 'C'], 'E': ['C']}
# 請找出使用者 'A' 的「二度好友」集合：
# 二度好友定義：A 的好友的好友，但「不能是 A 本人」，且「不能是 A 原本的一度好友」！
# 提示：利用集合的聯集與差集運算！

graph = {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A', 'D', 'E'], 'D': ['B', 'C'], 'E': ['C']}
user = 'A'

# 請在下方寫出你的解答代碼
direct_friends = set(graph.get(user, []))
fof_set = set()  # Friends of Friends

for f in direct_friends:
    for fof in graph.get(f, []):
        fof_set.add(fof)

# 扣除自己與原本的一度好友
second_degree = sorted((fof_set - direct_friends) - {user})

print("A 的二度好友:", second_degree)
# 預期結果: A 的二度好友: ['D', 'E']

## 10.8.4 多鍵綜合排序：字典轉串列反轉鍵值排序（免 lambda 技巧）

### 觀念說明
在競賽中，當我們用字典統計完數據後，最常見的後續需求是：
**「將資料依照出現次數（Value）由大到小排序；若次數相同，則依名稱（Key）由小到大排序」**。

許多人會說：這需要使用 `sort(key=lambda x: ...)`！
但是，自訂函式與 `lambda` 屬於第 11 與第 12 章的範疇。**在零提前依賴的原則下，初學者該如何優雅解題？**

### 答案就在 10-2 的「元組多欄位比大小」！
我們在 10-2 學過：Python 在比較元組時，會由左至右逐一比對：先比第 0 欄，若相同再比第 1 欄！
因此，我們只要使用**列表生成式**，將字典的鍵值對調打包成自訂評比元組：
```python
# 欲達到「次數由大到小（加負號 -count）、姓名由小到大（name）」
sorted_list = sorted([(-count, name) for name, count in tally.items()])
```
接著直接呼叫 `sorted()`，Python 就會完美依照我們的期望順序排列！完全不需要自訂函數，這就是資料結構原生特性的無窮智慧！

In [ ]:
# 10.8.4 範例：得票數降序、姓名升序多準則排序榜單
votes = {
    "Charlie": 5,
    "Alice": 8,
    "Bob": 8,
    "David": 3
}

# 透過元組反轉打包：(-票數, 姓名)
# -票數: 負號讓數值越大者負得越多，因此升序排序時會排在最前面（等同降序）！
# 姓名: 當票數相同時，依照字母 ASCII 正向排序（等同升序）！
packed = [(-count, name) for name, count in votes.items()]
sorted_packed = sorted(packed)

print("--- 最終英雄榜 (多條件排序) ---")
rank = 1
for neg_cnt, name in sorted_packed:
    real_cnt = -neg_cnt  # 負負得正還原真實票數
    print(f"第 {rank} 名: {name:<7} | 得票數: {real_cnt}")
    rank += 1

In [ ]:
# 10.8.4 填空題：商品庫存量由大到小排序
stock_dict = {"pen": 15, "book": 40, "eraser": 25}

# 任務：使用列表生成式將字典轉為 (庫存量, 商品名)，並使用 sorted(..., reverse=True) 降序排列
stock_tuples = [(qty, item) for item, qty in stock_dict.items()]
sorted_stock = sorted(stock_tuples, reverse=True)

# 請填入空格：
# stock_tuples = [(___, ___) for item, qty in stock_dict.items()]
print("庫存量降序排行:", sorted_stock)
# 預期輸出: 庫存量降序排行: [(40, 'book'), (25, 'eraser'), (15, 'pen')]

In [ ]:
# 10.8.4 練習題：英文單字出現頻率英雄榜
# 題目說明：
# 給定文章單字串列 essay_words = ["apple", "pie", "apple", "banana", "pie", "apple", "cat"]
# 1. 請先使用字典統計每個單字出現的次數。
# 2. 使用「元組反轉打包法」，將資料排序：
#    - 第一準則：出現次數由大到小（降序）。
#    - 第二準則：單字字母由小到大（升序）。
# 3. 依序輸出排序後的結果，格式為：<單字> : <次數>。
#
# 【公開測試資料 1】
# 輸入: ["apple", "pie", "apple", "banana", "pie", "apple", "cat"]
# 輸出:
# apple : 3
# pie : 2
# banana : 1
# cat : 1
#
# 【公開測試資料 2】
# 輸入: ["z", "a", "z", "b"]
# 輸出:
# z : 2
# a : 1
# b : 1

essay_words = ["apple", "pie", "apple", "banana", "pie", "apple", "cat"]

freq = {}
for w in essay_words:
    freq[w] = freq.get(w, 0) + 1

packed = [(-count, word) for word, count in freq.items()]
for neg_cnt, word in sorted(packed):
    print(word, ":", -neg_cnt)

In [ ]:
# 10.8.4 挑戰題：三欄位多準則複合排序
# 題目說明：
# 給定學生資料字典（姓名為鍵，值為 (年級, 分數) 元組）：
# students = {"Alice": (7, 90), "Bob": (8, 90), "Charlie": (7, 95), "David": (8, 85)}
# 請將學生排序，規則如下：
# 1. 分數最高者優先（降序）
# 2. 若分數相同，年級較小者優先（升序）
# 3. 若依然相同，姓名依字母升序
# 請輸出排在第一名的榜首學生姓名！

students = {"Alice": (7, 90), "Bob": (8, 90), "Charlie": (7, 95), "David": (8, 85)}

# 請在下方寫出你的解答代碼
# 打包元組: (-分數, 年級, 姓名)
packed = [(-score, grade, name) for name, (grade, score) in students.items()]
top_student = sorted(packed)[0][2]

print("榜首學生姓名:", top_student)
# 預期結果: 榜首學生姓名: Charlie

## 10.8.5 重複元素偵測與首個重複值定位（One-pass Hash Scan）

### 觀念說明
給定一個龐大的數據流，題目要求：**「找出數列中『第一個出現重複』的元素是什麼？」**

傳統的做法常常是：
- 雙重迴圈逐一比對 ➔ $O(N^2)$（遇到十萬筆測資直接超時 TLE）。
- 先排序再找相鄰相等 ➔ $O(N \log N)$，但排序會**打亂原本元素出現的時序順序**，導致找不到「誰先重複」！

### 單趟雜湊掃描（One-pass Hash Scan）
最頂級的解法，就是利用**集合邊走訪邊記錄**：
1. 建立一個空集合 `seen = set()`。
2. 依序走訪每一個元素 `x`：
   - **`if x in seen:`** ➔ 震撼命中！當前 `x` 就是全數列中**第一個重複出現的元素**，立刻 `break` 結束搜尋！
   - **`else:`** ➔ 將其加入留存：`seen.add(x)`。

整個過程只需要走訪一次，時間複雜度是極限的 **$O(N)$**，而且完美保持了時間序列的先後關係！

In [ ]:
# 10.8.5 範例：數據流中首個重複訪客瞬間鎖定
visitor_stream = ["U105", "U204", "U301", "U105", "U204", "U999"]

seen_visitors = set()
first_duplicate = None

for uid in visitor_stream:
    if uid in seen_visitors:
        first_duplicate = uid
        print(f"Bingo! 找到首個重複訪問者: {uid}")
        break
    seen_visitors.add(uid)

if first_duplicate is None:
    print("全流無重複訪客")

In [ ]:
# 10.8.5 填空題：字串中首個重複字母偵測
text = "abcdcba"

seen_chars = set()
first_repeat_char = None

for ch in text:
    if ch in seen_chars:
        first_repeat_char = ch
        break
    seen_chars.add(ch)

# 請填入空格：
# if ch in ___:
#     first_repeat_char = ch
#     break
# seen_chars.add(___)
print("第一個重複字母:", first_repeat_char)
# 預期輸出: 第一個重複字母: c

In [ ]:
# 10.8.5 練習題：首個重複整數與索引定位器
# 題目說明：
# 給定一整數串列 numbers。
# 請在單趟 O(N) 走訪中，找出「第一個重複出現的數字」，以及該數字「第二次出現時的索引值（0-based）」。
# 若全程皆無重複，輸出 "ALL_UNIQUE"。
#
# 【公開測試資料 1】
# 數列: [10, 20, 30, 40, 20, 50]
# 輸出: 重複數值: 20 , 索引: 4
#
# 【公開測試資料 2】
# 數列: [1, 2, 3, 4, 5]
# 輸出: ALL_UNIQUE

numbers = [10, 20, 30, 40, 20, 50]

seen = set()
found_val = None
found_idx = -1

for idx in range(len(numbers)):
    num = numbers[idx]
    if num in seen:
        found_val = num
        found_idx = idx
        break
    seen.add(num)

if found_val is not None:
    print(f"重複數值: {found_val} , 索引: {found_idx}")
else:
    print("ALL_UNIQUE")

In [ ]:
# 10.8.5 挑戰題：首個出現滿 K 次的元素判定
# 題目說明：
# 給定整數串列 stream = [2, 3, 5, 3, 2, 4, 3, 2] 以及門檻 k = 3
# 請使用字典統計，找出「第一個達到累積出現 k 次」的幸運數字，並在達到當下立即終止迴圈輸出！

stream = [2, 3, 5, 3, 2, 4, 3, 2]
k = 3

# 請在下方寫出你的解答代碼
counts = {}
winner = None

for x in stream:
    counts[x] = counts.get(x, 0) + 1
    if counts[x] == k:
        winner = x
        break

print(f"首個達到 {k} 次的數字:", winner)
# 預期結果: 首個達到 3 次的數字: 3

## 10.8.6 APCS 歷屆真題中的雜湊思維總覽（為第 14 章 g276、c291 深度鋪墊）

### 觀念說明
恭喜你！走到了這裡，你已經具備了征服 APCS 實作題最關鍵的武器庫！
讓我們回顧在歷屆 APCS 實作真題中，雜湊容器是如何大顯身手的：

1. **APCS g276 魔王迷宮（中級題 / 舊版實作第 2 題）**：
   多隻魔王在棋盤上移動並留下炸彈，踩中炸彈則爆炸引爆魔王消失。
   - 炸彈位置以座標元組 `(r, c)` 放入**集合 `bombs = set()`**。
   - 多隻魔王移動後，踩中炸彈的引爆消除，直接使用**集合差集 `bombs - exploded`** 一行搞定！

2. **APCS c291 小群體（中級題 / 舊版實作第 2 題）**：
   $N$ 個人互相指名好友，求形成封閉循環的群體總數。
   - 以**字典或串列**儲存好友關係圖。
   - 以**已走訪集合 `visited = set()`** 追蹤循環軌跡，避免重複拜訪與無窮迴圈！

雜湊思維的本質就是：**「用記憶體空間換取極致的 $O(1)$ 時間」**。只要熟練這套打法，任何複雜的模擬與搜尋題都能迎刃而解！

In [ ]:
# 10.8.6 範例：APCS g276 魔王迷宮炸彈引爆原型模擬
# 初始已放置的炸彈座標集合
bombs = {(1, 1), (2, 3), (3, 3), (5, 5)}

# 本回合魔王們移動後所踩到的座標串列
monsters_pos = [(2, 3), (4, 4), (5, 5)]

print("回合開始時全地圖炸彈:", bombs)

# 找出被踩中的炸彈（交集）
triggered = bombs & set(monsters_pos)
print("本回合被引爆的炸彈:", triggered)

# 從地圖中移除被引爆的炸彈（差集）
bombs = bombs - triggered
print("引爆後剩餘未爆炸彈:", bombs)
print("殘留炸彈數量:", len(bombs))

In [ ]:
# 10.8.6 填空題：APCS c291 朋友圈走訪標記原型
friends = {0: 1, 1: 2, 2: 0, 3: 4, 4: 3}  # 0->1->2->0 (環1), 3->4->3 (環2)

visited = set()
group_count = 0

for person in friends:
    if person not in visited:
        group_count += 1
        curr = person
        while curr not in visited:
            visited.add(curr)
            curr = friends[curr]

# 請將上方循環加入填入空格：
# while curr not in ___:
#     visited.add(___)
#     curr = friends[___]
print("獨立朋友圈群體總數:", group_count)
# 預期輸出: 獨立朋友圈群體總數: 2

In [ ]:
# 10.8.6 練習題：APCS c291 小群體演算法完整實作
# 題目說明：
# 給定 N 個人的好友指涉清單 relations（長度為 N，relations[i] 代表編號 i 的人的好友）。
# 題目保證每個人恰有一個好友，因此全體人員一定會形成若干個封閉的獨立圈子（環）。
# 請使用集合 visited = set() 來追蹤走訪過的成員，計算共有幾個獨立小群體。
#
# 【公開測試資料 1】
# 好友關係: [1, 2, 0, 4, 3]  (0->1->2->0, 3->4->3 共 2 群)
# 輸出: 小群體數量: 2
#
# 【公開測試資料 2】
# 好友關係: [2, 0, 1]  (0->2->1->0 共 1 群)
# 輸出: 小群體數量: 1

relations = [1, 2, 0, 4, 3]

visited = set()
num_groups = 0

for i in range(len(relations)):
    if i not in visited:
        num_groups += 1
        curr = i
        while curr not in visited:
            visited.add(curr)
            curr = relations[curr]

print("小群體數量:", num_groups)

In [ ]:
# 10.8.6 挑戰題：網格多物件移動碰撞消除模擬
# 題目說明：
# 網格上有 4 顆粒子，每顆粒子的初始位置與速度向量為 ((r, c), (dr, dc))：
# particles = [
#     ((0, 0), (1, 1)),
#     ((2, 0), (-1, 1)),
#     ((0, 2), (1, -1)),
#     ((5, 5), (0, 0))
# ]
# 模擬移動「1 回合」後：
# 1. 計算每顆粒子移動後的新座標 (r + dr, c + dc)。
# 2. 若兩顆或多顆粒子在新座標相撞（座標相同），則這些相撞的粒子全部爆炸消除！
# 請印出 1 回合後剩餘未相撞的粒子數量。

particles = [
    ((0, 0), (1, 1)),
    ((2, 0), (-1, 1)),
    ((0, 2), (1, -1)),
    ((5, 5), (0, 0))
]

# 請在下方寫出你的解答代碼
new_positions = []
for (r, c), (dr, dc) in particles:
    new_positions.append((r + dr, c + dc))

# 統計各座標的粒子數量
pos_counts = {}
for pos in new_positions:
    pos_counts[pos] = pos_counts.get(pos, 0) + 1

# 只有數量為 1 的代表沒相撞
survived = [p for p in new_positions if pos_counts[p] == 1]

print("未相撞存活粒子數量:", len(survived))
# 預期結果: 未相撞存活粒子數量: 1 （座標 (1, 1) 有三顆粒子相撞消除，只剩 (5, 5) 存活）

## 10.8 學習總結與第十章大圓滿檢核

🎉 **賀！你已經完整通關了第十章「字典、集合與元組」的所有 8 節、48 個微型子單元！**

### 雜湊容器三大天王終極整合表
| 資料結構 | 核心特性 | 雜湊限制 | APCS 終極必殺技 |
| :--- | :--- | :--- | :--- |
| **元組 (Tuple)** | 不可變、有序、輕量、支援解包 | 元素可雜湊時自己就可雜湊 | `(r, c)` 二維網格座標封裝、多準則排序評比鍵 |
| **字典 (Dict)** | 鍵唯一、鍵值映射、動態擴展 | 鍵（Key）必須可雜湊 | 動態頻率計數、稀疏網格、圖論鄰接串列、座標壓縮 |
| **集合 (Set)** | 元素唯一、無序、天然去重 | 元素必須可雜湊 | 范氏圖代數運算（`&`, `|`, `-`, `^`）、$O(1)$ 足跡標記、黑名單消除 |

現在，你已經完全具備了中高階資料結構的設計與抽象思維能力！在接下來的**第十一章**中，我們將正式進入**「自訂函數（Function）與變數作用域（Scope）」**，學習如何將這套強大的邏輯封裝為模組化黑盒子，向更高階的演算法殿堂邁進！